In [1]:
import os
import pandas as pd
from pathlib import Path
from transformers import pipeline

print(" Imports done!")

 Imports done!


In [2]:
BASE_DIR   = Path(os.getcwd()).parent
INPUT_CSV  = BASE_DIR / "data" / "processed" / "emails_with_intent.csv"
OUTPUT_CSV = BASE_DIR / "data" / "processed" / "emails_with_sentiment.csv"

print("Input :", INPUT_CSV)
print("Output:", OUTPUT_CSV)

Input : d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_with_intent.csv
Output: d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_with_sentiment.csv


In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} emails")
df.head(3)

Loaded 8,064 emails


,sender,to,subject,date,body_clean,body_processed,intent
0,phillip.allen@enron.com,john.lavorato@enron.com,Re:,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",Traveling to have a business meeting takes the...,traveling business meeting take fun trip espec...,meeting_request
1,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)","Randy,\n\n Can you send me a schedule of the s...",randy send schedule salary level everyone sche...,meeting_request
2,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,"Thu, 31 Aug 2000 04:17:00 -0700 (PDT)","Greg,\n\n How about either next Tuesday or Thu...",greg either next tuesday thursday phillip,greeting


In [4]:
print("Loading sentiment model")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512
)
print(" Model loaded!")

Loading sentiment model


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2171.21it/s]


 Model loaded!


In [5]:
def get_sentiment(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return "NEUTRAL"
    try:
        result = sentiment_model(text[:512])[0]
        return result['label']
    except:
        return "NEUTRAL"

print("Function ready!")

Function ready!


In [6]:
print(" Analyzing sentiment.")
df['sentiment'] = df['body_clean'].apply(get_sentiment)
print("Done!")

print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())

 Analyzing sentiment.
Done!

Sentiment Distribution:
sentiment
NEGATIVE    5070
POSITIVE    2994
Name: count, dtype: int64


In [7]:
print("Intent vs Sentiment:\n")
print(pd.crosstab(df['intent'], df['sentiment']))

Intent vs Sentiment:

sentiment            NEGATIVE  POSITIVE
intent                                 
apology                    80        16
appreciation              587       801
complaint                 230        86
farewell                    3         5
general                   775       275
greeting                  397       201
information_request      1048       317
meeting_request          1950      1293


In [8]:
df.to_csv(OUTPUT_CSV, index=False)
print(f" Saved → {OUTPUT_CSV}")
print(f" Total rows: {len(df):,}")

 Saved → d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_with_sentiment.csv
 Total rows: 8,064
